# MambaIR-Scratch: SIDL Scratch 복원 프로젝트
**구성**: MambaIR 베이스 + DirectionalSSM + ScratchMaskAttn + ScratchLoss

**실행 순서**: 셀을 위에서 아래로 순서대로 실행하세요.

## 1. Google Drive 마운트 & MambaIR 클론

In [1]:
# 1. 찌꺼기가 남은 기존 유령 폴더를 강제로 완전 삭제
!rm -rf /content/drive

# 2. 깨끗해진 상태에서 다시 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive

# 3. MambaIR 레포 클론 (처음 실행 시)
import os
if not os.path.exists('/content/drive/MyDrive/MambaIR'):
    !git clone https://github.com/csguoh/MambaIR
    print('클론 완료')
else:
    print('이미 존재, 스킵')

%cd /content/drive/MyDrive/MambaIR

Mounted at /content/drive
/content/drive/MyDrive
이미 존재, 스킵
/content/drive/MyDrive/MambaIR


## 2. 환경 설치 (처음 1회만)

In [2]:

!pip install basicsr==1.4.2 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.5/172.5 kB 16.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 129.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 30.1 MB/s eta 0:00:00


In [3]:
# mamba-ssm 직접 설치
!pip install mamba-ssm --no-build-isolation -q
!pip install causal-conv1d --no-build-isolation -q
!pip install einops -q

# MambaIR setup만 따로 (requirements.txt 스킵)
import os
os.chdir('/content/drive/MyDrive/MambaIR')
!python setup.py develop --no_cuda_ext 2>/dev/null || echo "setup 완료"

print("설치 완료")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.4/216.4 kB 18.4 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.7/327.7 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.3/29.3 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.8/897.8 kB 67.3 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
setup 완료
설치 완료


## 3. 커스텀 파일 배치
> Drive 에 올려둔 `mambair_scratch_arch.py` 와 `scratch_losses.py` 를
> MambaIR 레포의 올바른 위치에 복사합니다.

In [4]:
import shutil, os

# ── 파일 경로 설정 ──
# Drive 에 업로드한 커스텀 파일 위치
ARCH_SRC  = '/content/drive/MyDrive/SIDL_custom/mambair_scratch_arch.py'
LOSS_SRC  = '/content/drive/MyDrive/SIDL_custom/scratch_losses.py'

# MambaIR 레포 내 배치 위치
ARCH_DST  = '/content/drive/MyDrive/MambaIR/basicsr/archs/mambair_scratch_arch.py'
LOSS_DST  = '/content/drive/MyDrive/MambaIR/basicsr/losses/scratch_losses.py'

shutil.copy(ARCH_SRC, ARCH_DST)
shutil.copy(LOSS_SRC, LOSS_DST)
print('파일 복사 완료')

# ── __init__.py 에 import 추가 ──
arch_init  = '/content/drive/MyDrive/MambaIR/basicsr/archs/__init__.py'
loss_init  = '/content/drive/MyDrive/MambaIR/basicsr/losses/__init__.py'

def add_import_if_missing(path, import_line):
    with open(path, 'r') as f:
        content = f.read()
    if import_line not in content:
        with open(path, 'a') as f:
            f.write('\n' + import_line + '\n')
        print(f'추가됨: {import_line}')
    else:
        print(f'이미 존재: {import_line}')

add_import_if_missing(arch_init, 'from .mambair_scratch_arch import MambaIRScratch')
add_import_if_missing(loss_init, 'from .scratch_losses import ScratchLoss')

파일 복사 완료
이미 존재: from .mambair_scratch_arch import MambaIRScratch
이미 존재: from .scratch_losses import ScratchLoss


## 4. 학습 YAML 생성

### 빠른 로컬 세팅용
: 앞으로 코랩을 껐다 켤 때마다 실행해야 하는 셀

In [7]:
# 1. 드라이브의 압축 파일을 코랩 자체 초고속 디스크로 복사
!cp /content/drive/MyDrive/scratch_dataset.zip /content/

# 2. 코랩 로컬 디스크에 압축 해제
!unzip -q /content/scratch_dataset.zip -d /content/local_dataset/
print("로컬 가상 디스크에 초고속 데이터 세팅 완료!")

로컬 가상 디스크에 초고속 데이터 세팅 완료!


### 학습 yaml

In [ ]:
import os

task       = 'scratch'
difficulty = 'hard'
width      = 48
num_blocks = 8
total_iter = 200000   # 🎯 20만 번으로 세팅 완료!

out_dir = '/content/drive/MyDrive/sidl_options/train'
os.makedirs(out_dir, exist_ok=True)

template = f"""\
name: MambaIRScratch-SIDL-{task}-dim{width}-V2  # 💡 V2로 이름 변경 (새로운 폴더에 독립 저장)
model_type: SRModel
scale: 1
num_gpu: 1
manual_seed: 42

path:
  root: /content/drive/MyDrive/MambaIR_experiments
  pretrain_network_g: ~
  strict_load_g: true
  resume_state: ~       # 💡 0번부터 새롭게 시작

datasets:
  train:
    name: sidl-{task}-train
    type: PairedImageDataset
    dataroot_gt: /content/local_dataset/train/{task}/target
    dataroot_lq: /content/local_dataset/train/{task}/input
    io_backend:
      type: disk
    gt_size: 64
    use_hflip: true
    use_rot: true
    use_shuffle: true
    num_worker_per_gpu: 2
    batch_size_per_gpu: 4
  val:
    name: sidl-{task}-{difficulty}-val
    type: PairedImageDataset
    dataroot_gt: /content/local_dataset/val/{task}/{difficulty}/target
    dataroot_lq: /content/local_dataset/val/{task}/{difficulty}/input
    io_backend:
      type: disk

network_g:
  type: MambaIRScratch
  inp_channels: 3
  out_channels: 3
  dim: {width}
  num_blocks: {num_blocks}
  d_state: 16
  d_conv: 4
  expand: 2
  img_range: 1.0

train:
  optim_g:
    type: Adam
    lr: !!float 1e-4        # 💡 과부하를 막기 위해 안정적인 속도 설정
    weight_decay: !!float 1e-4
    betas: [0.9, 0.999]
  scheduler:
    type: CosineAnnealingLR # 💡 충돌 없는 가장 안정적인 스케줄러
    T_max: {total_iter}
    eta_min: !!float 1e-7
  total_iter: {total_iter}
  warmup_iter: -1
  pixel_opt:
    type: ScratchLoss
    loss_weight: 1.0
    w_char: 1.0
    w_edge: 0.1
    w_ssim: 0.1
    w_fft: 0.1            # 💡 V2의 핵심: 푸리에 변환 주파수 손실 활성화!
    eps: !!float 1e-3

val:
  val_freq: !!float 2000
  save_img: true
  metrics:
    psnr:
      type: calculate_psnr
      crop_border: 0
      test_y_channel: false
    ssim:
      type: calculate_ssim
      crop_border: 0
      test_y_channel: false

logger:
  print_freq: 10
  save_checkpoint_freq: !!float 2000
  use_tb_logger: true
  wandb:
    project: ~
    resume_id: ~

dist_params:
  backend: nccl
  port: 29500
"""

out_path = os.path.join(out_dir, f'MambaIRScratch_{task}.yml')
with open(out_path, 'w') as f:
    f.write(template)

print(f'✅ 완벽한 V2 YAML 생성 완료: {out_path}')

✅ 완벽한 V2 YAML 생성 완료: /content/drive/MyDrive/sidl_options/train/MambaIRScratch_scratch.yml


In [ ]:
# import os

# task       = 'scratch'
# difficulty = 'hard'
# width      = 48
# num_blocks = 8
# total_iter = 10000  # 테스트 후 200000으로 변경

# out_dir = '/content/drive/MyDrive/sidl_options/train'
# os.makedirs(out_dir, exist_ok=True)

# template = f"""\
# name: MambaIRScratch-SIDL-{task}-dim{width}
# model_type: SRModel
# scale: 1
# num_gpu: 1
# manual_seed: 42

# path:
#   root: /content/drive/MyDrive/MambaIR_experiments
#   pretrain_network_g: ~
#   strict_load_g: true
#   resume_state: ~

# datasets:
#   train:
#     name: sidl-{task}-train
#     type: PairedImageDataset
#     dataroot_gt: /content/local_dataset/train/{task}/target
#     dataroot_lq: /content/local_dataset/train/{task}/input
#     io_backend:
#       type: disk
#     gt_size: 64
#     use_hflip: true
#     use_rot: true
#     use_shuffle: true
#     num_worker_per_gpu: 2
#     batch_size_per_gpu: 4
#   val:
#     name: sidl-{task}-{difficulty}-val
#     type: PairedImageDataset
#     dataroot_gt: /content/local_dataset/val/{task}/{difficulty}/target
#     dataroot_lq: /content/local_dataset/val/{task}/{difficulty}/input
#     io_backend:
#       type: disk

# network_g:
#   type: MambaIRScratch
#   inp_channels: 3
#   out_channels: 3
#   dim: {width}
#   num_blocks: {num_blocks}
#   d_state: 16
#   d_conv: 4
#   expand: 2
#   img_range: 1.0

# train:
#   optim_g:
#     type: Adam
#     # lr: !!float 2e-4
#     lr: !!float 1e-4
#     weight_decay: !!float 1e-4
#     betas: [0.9, 0.999]
#   scheduler:
#     # type: CosineAnnealingLR
#     type: CosineAnnealingRestartLR
#     T_max: {total_iter}
#     eta_min: !!float 1e-7
#   total_iter: {total_iter}
#   warmup_iter: -1
#   pixel_opt:
#     type: ScratchLoss
#     loss_weight: 1.0
#     w_char: 1.0
#     w_edge: 0.1
#     w_ssim: 0.1
#     eps: !!float 1e-3

# val:
#   val_freq: !!float 2000
#   save_img: true
#   metrics:
#     psnr:
#       type: calculate_psnr
#       crop_border: 0
#       test_y_channel: false
#     ssim:
#       type: calculate_ssim
#       crop_border: 0
#       test_y_channel: false

# logger:
#   print_freq: 10
#   save_checkpoint_freq: !!float 2000
#   use_tb_logger: true
#   wandb:
#     project: ~
#     resume_id: ~

# dist_params:
#   backend: nccl
#   port: 29500
# """

# out_path = os.path.join(out_dir, f'MambaIRScratch_{task}.yml')
# with open(out_path, 'w') as f:
#     f.write(template)

# print(f'YAML 생성 완료: {out_path}')

YAML 생성 완료: /content/drive/MyDrive/sidl_options/train/MambaIRScratch_scratch.yml


## 5. 모델 동작 확인 (학습 전 sanity check)

In [ ]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/MambaIR')

import torch
from basicsr.archs.mambair_scratch_arch import MambaIRScratch
from basicsr.losses.scratch_losses import ScratchLoss

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# 모델 빌드
model = MambaIRScratch(
    inp_channels=3, out_channels=3,
    dim=48, num_blocks=8
).to(device)

# 파라미터 수
params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Parameters: {params:.2f} M')

# Forward pass 확인
dummy_lq = torch.rand(1, 3, 64, 64).to(device)
dummy_gt = torch.rand(1, 3, 64, 64).to(device)
with torch.no_grad():
    out = model(dummy_lq)
print(f'Output shape: {out.shape}')

# Loss 확인
loss_fn = ScratchLoss(w_char=1.0, w_edge=0.1, w_ssim=0.1).to(device)
loss = loss_fn(out, dummy_gt)
print(f'Loss: {loss.item():.4f}')
print('Sanity check 통과!')

Device: cuda
Parameters: 0.31 M
Output shape: torch.Size([1, 3, 64, 64])
Loss: 0.4106
Sanity check 통과!


## 6. 학습 실행

torch vision 패치

In [ ]:
# torchvision 호환 패치 (파일 직접 수정)
import site
import os

# basicsr degradations.py 경로 찾기
for path in site.getsitepackages():
    target = os.path.join(path, 'basicsr/data/degradations.py')
    if os.path.exists(target):
        with open(target, 'r') as f:
            content = f.read()
        # 구버전 import를 새버전으로 교체
        content = content.replace(
            'from torchvision.transforms.functional_tensor import rgb_to_grayscale',
            'from torchvision.transforms.functional import rgb_to_grayscale'
        )
        with open(target, 'w') as f:
            f.write(content)
        print(f'패치 완료: {target}')
        break

base_model 패치

In [ ]:
path = '/content/drive/MyDrive/MambaIR/basicsr/models/base_model.py'

with open(path, 'r') as f:
    content = f.read()

# 잘못된 패치 제거하고 올바르게 교체
old = """        else:
            if scheduler_type == 'CosineAnnealingLR':
                schedulers.append(
                    CosineAnnealingLR(
                        self.optimizer_g,
                        T_max=train_opt['scheduler']['T_max'],
                        eta_min=train_opt['scheduler'].get('eta_min', 0)
                    )
                )
            else:
                raise NotImplementedError(f'Scheduler {scheduler_type} is not implemented yet.')"""

new = """        elif scheduler_type == 'CosineAnnealingLR':
            for optimizer in self.optimizers:
                self.schedulers.append(
                    torch.optim.lr_scheduler.CosineAnnealingLR(
                        optimizer,
                        T_max=train_opt['scheduler']['T_max'],
                        eta_min=train_opt['scheduler'].get('eta_min', 0)
                    )
                )
        else:
            raise NotImplementedError(f'Scheduler {scheduler_type} is not implemented yet.')"""

content = content.replace(old, new)

with open(path, 'w') as f:
    f.write(content)

print('패치 완료')

패치 완료


학습 실행

In [ ]:
%cd /content/drive/MyDrive/MambaIR

!torchrun \
  --standalone \
  --nnodes=1 \
  --nproc_per_node=1 \
  --master_port=4321 \
  basicsr/train.py \
    -opt /content/drive/MyDrive/sidl_options/train/MambaIRScratch_scratch.yml \
    --launcher pytorch

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
2026-06-02 17:00:00,254 INFO: [Mamba..][epoch:  9, iter:  25,940, lr:(9.591e-05,)] [eta: 7:21:49, time (data): 0.065 (0.002)] l_pix: 3.7047e-02 
2026-06-02 17:00:00,904 INFO: [Mamba..][epoch:  9, iter:  25,950, lr:(9.591e-05,)] [eta: 7:21:42, time (data): 0.065 (0.002)] l_pix: 3.2156e-01 
2026-06-02 17:00:01,563 INFO: [Mamba..][epoch:  9, iter:  25,960, lr:(9.590e-05,)] [eta: 7:21:35, time (data): 0.065 (0.002)] l_pix: 1.3024e-01 
2026-06-02 17:00:02,210 INFO: [Mamba..][epoch:  9, iter:  25,970, lr:(9.590e-05,)] [eta: 7:21:27, time (data): 0.065 (0.002)] l_pix: 2.5916e-01 
2026-06-02 17:00:02,858 INFO: [Mamba..][epoch:  9, iter:  25,980, lr:(9.590e-05,)] [eta: 7:21:20, time (data): 0.065 (0.002)] l_pix: 4.6027e-02 
2026-06-02 17:00:03,502 INFO: [Mamba..][epoch:  9, iter:  25,990, lr:(9.590e-05,)] [eta: 7:21:13, time (data): 0.065 (0.002)] l_pix: 4.4387e-01 
2026-06-02 17:00:04,147 INFO: [Mamba..][epoch:  9, iter:  26,000, lr:(9.589e-05,)] [eta: 7:21:

## 7. 추론 YAML 생성 (easy/medium/hard 3개)

In [16]:
import os

task          = 'scratch'
width         = 48
num_blocks    = 8
difficulty_levels = ['easy', 'medium', 'hard']

# 체크포인트 경로 (학습 완료 후 실제 경로로 수정)
ckpt_path = '/content/drive/MyDrive/MambaIR/experiments/MambaIRScratch-SIDL-scratch-dim48-V2/models/net_g_74000.pth'
# ckpt_path = '/content/drive/MyDrive/MambaIR/experiments/MambaIRScratch-SIDL-scratch-dim48/models/net_g_200000.pth'

out_dir = '/content/drive/MyDrive/sidl_options/test'
os.makedirs(out_dir, exist_ok=True)

for difficulty in difficulty_levels:
    template = f"""\
name: MambaIRScratch-SIDL-{task}-{difficulty}-test
# model_type: ImageRestorationModel
model_type: SRModel
scale: 1
num_gpu: 1
manual_seed: 42

path:
  root: /content/drive/MyDrive/MambaIR_results
  pretrain_network_g: {ckpt_path}
  strict_load_g: true
  resume_state: ~

datasets:
  test:
    name: sidl-{task}-{difficulty}-test
    type: PairedImageDataset
    dataroot_gt: /content/drive/MyDrive/test/{task}/{difficulty}/input
    dataroot_lq: /content/drive/MyDrive/test/{task}/{difficulty}/input
    io_backend:
      type: disk

network_g:
  type: MambaIRScratch
  inp_channels: 3
  out_channels: 3
  dim: {width}
  num_blocks: {num_blocks}
  d_state: 16
  d_conv: 4
  expand: 2
  img_range: 1.0

val:
  save_img: true
  suffix: ~
  metrics:
    psnr:
      type: calculate_psnr
      crop_border: 0
      test_y_channel: false
    ssim:
      type: calculate_ssim
      crop_border: 0
      test_y_channel: false

dist_params:
  backend: nccl
  port: 29500
"""
    out_path = os.path.join(out_dir, f'MambaIRScratch_{task}_{difficulty}.yml')
    with open(out_path, 'w') as f:
        f.write(template)
    print(f'생성: {out_path}')

생성: /content/drive/MyDrive/sidl_options/test/MambaIRScratch_scratch_easy.yml
생성: /content/drive/MyDrive/sidl_options/test/MambaIRScratch_scratch_medium.yml
생성: /content/drive/MyDrive/sidl_options/test/MambaIRScratch_scratch_hard.yml


## 8. 추론 실행 (Easy / Medium / Hard)

In [17]:
import site
import os

# basicsr degradations.py 경로 찾기 및 구버전 import 패치
for path in site.getsitepackages():
    target = os.path.join(path, 'basicsr/data/degradations.py')
    if os.path.exists(target):
        with open(target, 'r') as f:
            content = f.read()

        content = content.replace(
            'from torchvision.transforms.functional_tensor import rgb_to_grayscale',
            'from torchvision.transforms.functional import rgb_to_grayscale'
        )

        with open(target, 'w') as f:
            f.write(content)
        print(f'패치 완료: {target}')
        break

패치 완료: /usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py


In [18]:
%cd /content/drive/MyDrive/MambaIR

for diff in ['easy', 'medium', 'hard']:
    print(f'========== Testing: {diff} ==========')
    !torchrun --standalone \
      --nnodes=1 \
      --nproc_per_node=1 \
      --master_port=4321 \
      basicsr/test.py \
      -opt /content/drive/MyDrive/sidl_options/test/MambaIRScratch_scratch_{diff}.yml \
      --launcher pytorch

/content/drive/MyDrive/MambaIR
========== Testing: easy ==========
Path already exists. Rename it to /content/drive/MyDrive/MambaIR/results/MambaIRScratch-SIDL-scratch-easy-test_archived_20260603_120841
2026-06-03 12:08:41,681 INFO: 
  name: MambaIRScratch-SIDL-scratch-easy-test
  model_type: SRModel
  scale: 1
  num_gpu: 1
  manual_seed: 42
  path:[
    root: /content/drive/MyDrive/MambaIR_results
    pretrain_network_g: /content/drive/MyDrive/MambaIR/experiments/MambaIRScratch-SIDL-scratch-dim48-V2/models/net_g_74000.pth
    strict_load_g: True
    resume_state: None
    results_root: /content/drive/MyDrive/MambaIR/results/MambaIRScratch-SIDL-scratch-easy-test
    log: /content/drive/MyDrive/MambaIR/results/MambaIRScratch-SIDL-scratch-easy-test
    visualization: /content/drive/MyDrive/MambaIR/results/MambaIRScratch-SIDL-scratch-easy-test/visualization
  ]
  datasets:[
    test:[
      name: sidl-scratch-easy-test
      type: PairedImageDataset
      dataroot_gt: /content/drive/MyDri

### 결과시각화


In [ ]:
import os
import glob
import matplotlib.pyplot as plt
from PIL import Image

# 난이도 설정 (확인하고 싶은 난이도로 변경 가능: 'easy', 'medium', 'hard')
diff = 'easy'

result_dir = f'/content/drive/MyDrive/MambaIR/results/MambaIRScratch-SIDL-scratch-{diff}-test/visualization/sidl-scratch-{diff}-test'
input_dir = f'/content/drive/MyDrive/test/scratch/{diff}/input'
target_dir = f'/content/drive/MyDrive/test/scratch/{diff}/target'

if not os.path.exists(result_dir):
    print(f"🚨 아직 추론 결과가 없습니다.\n경로: {result_dir}")
else:
    result_images = glob.glob(os.path.join(result_dir, '*.png'))

    if not result_images:
        print("🚨 폴더는 있지만 이미지가 없습니다.")
    else:
        res_path = result_images[151]
        img_name = os.path.basename(res_path)

        # 💡 핵심 수정: 뒤에 붙은 긴 꼬리표를 잘라내고 진짜 원본 이름만 추출!
        orig_name = img_name.replace(f'_MambaIRScratch-SIDL-scratch-{diff}-test', '')

        inp_path = os.path.join(input_dir, orig_name)
        tgt_path = os.path.join(target_dir, orig_name)

        fig, axes = plt.subplots(1, 3, figsize=(18, 6))

        # 1. 원본 입력 (상처 난 이미지)
        if os.path.exists(inp_path):
            axes[0].imshow(Image.open(inp_path))
        else:
            axes[0].text(0.5, 0.5, 'Input Not Found', ha='center', fontsize=12)
        axes[0].set_title('1. Input (Scratch)', fontsize=16, fontweight='bold')
        axes[0].axis('off')

        # 2. 복원 결과 (MambaIR 모델)
        axes[1].imshow(Image.open(res_path))
        axes[1].set_title('2. Restored (MambaIR)', fontsize=16, fontweight='bold', color='blue')
        axes[1].axis('off')

        # 3. 정답 타겟 (깨끗한 이미지)
        if os.path.exists(tgt_path):
            axes[2].imshow(Image.open(tgt_path))
        else:
            axes[2].text(0.5, 0.5, 'Target Not Found', ha='center', fontsize=12)
        axes[2].set_title('3. Target (Clean)', fontsize=16, fontweight='bold')
        axes[2].axis('off')

        plt.tight_layout()
        plt.show()

        print(f"✅ 원본 파일명: {orig_name}")
        print(f"✅ 복원 파일명: {img_name}")

## 9. 제출 파일 생성 (.tar)

In [ ]:
# import os, shutil, glob, tarfile

# task = 'scratch'
# difficulty_levels = ['easy', 'medium', 'hard']
# # 베이스 경로 수정 완료!
# base_path = '/content/drive/MyDrive/MambaIR/results'

# source_dirs = {
#     diff: os.path.join(
#         base_path,
#         f'MambaIRScratch-SIDL-{task}-{diff}-test/visualization/sidl-{task}-{diff}-test'
#     )
#     for diff in difficulty_levels
# }

# temp_dir = '/content/temp_submission'
# os.makedirs(temp_dir, exist_ok=True)

# task_dir = os.path.join(temp_dir, task)
# os.makedirs(task_dir, exist_ok=True)

# for diff in difficulty_levels:
#     input_dir = os.path.join(task_dir, diff, 'input')
#     os.makedirs(input_dir, exist_ok=True)

#     png_files = glob.glob(os.path.join(source_dirs[diff], '*.png'))
#     non_gt = [f for f in png_files if not f.endswith('_gt.png')]

#     for f_path in non_gt:
#         shutil.copy(f_path, input_dir)

# tar_path = '/content/drive/MyDrive/sidl_submission.tar'
# with tarfile.open(tar_path, "w") as tar:
#     tar.add(task_dir, arcname=task)

# print(f"✅ 제출용 압축 파일 생성 완료: {tar_path}")

In [15]:
import os, shutil, glob, tarfile

task = 'scratch'
difficulty_levels = ['easy', 'medium', 'hard']
base_path = '/content/drive/MyDrive/MambaIR/results'

source_dirs = {
    diff: os.path.join(
        base_path,
        f'MambaIRScratch-SIDL-{task}-{diff}-test/visualization/sidl-{task}-{diff}-test'
    )
    for diff in difficulty_levels
}

# 기존 찌꺼기와 섞이지 않게 새로운 임시 폴더 사용
temp_dir = '/content/temp_submission_fixed_2'
os.makedirs(temp_dir, exist_ok=True)

task_dir = os.path.join(temp_dir, task)
os.makedirs(task_dir, exist_ok=True)

for diff in difficulty_levels:
    input_dir = os.path.join(task_dir, diff, 'input')
    os.makedirs(input_dir, exist_ok=True)

    png_files = glob.glob(os.path.join(source_dirs[diff], '*.png'))

    for f_path in png_files:
        filename = os.path.basename(f_path)

        # 💡 핵심: 압축할 때 파일 이름에서 긴 꼬리표를 싹둑 잘라냅니다!
        clean_name = filename.replace(f'_MambaIRScratch-SIDL-scratch-{diff}-test', '')

        dest_path = os.path.join(input_dir, clean_name)
        shutil.copy(f_path, dest_path)

# 새로운 이름으로 압축 파일 생성
tar_path = '/content/drive/MyDrive/sidl_submission_fixed.tar'
with tarfile.open(tar_path, "w") as tar:
    tar.add(task_dir, arcname=task)

print(f"✅ 제출용 압축 파일 생성 완료: {tar_path}")

KeyboardInterrupt: 

In [ ]:
import sys
# MambaIR 폴더를 못 찾을까 봐 안전장치로 경로를 한 번 더 잡아줍니다.
sys.path.insert(0, '/content/drive/MyDrive/MambaIR')

import torch
from thop import profile
# 💡 핵심 수정: 경로와 파일명을 질문자님의 세팅에 맞게 정확히 고쳤습니다!
from basicsr.archs.mambair_scratch_arch import MambaIRScratch

# 테스트용 빈 이미지 (예: 3채널 256x256 해상도)
dummy_input = torch.randn(1, 3, 256, 256).cuda()

# 모델 불러오기
model = MambaIRScratch(inp_channels=3, out_channels=3, dim=48, num_blocks=8).cuda()

# GMACs 측정
macs, params = profile(model, inputs=(dummy_input, ))
print(f"✅ 연산량(GMACs): {macs / 10**9:.2f} G")

In [19]:
import os
import glob

ckpt_dir = '/content/drive/MyDrive/MambaIR/experiments/MambaIRScratch-SIDL-scratch-dim48/models'
files = sorted(glob.glob(os.path.join(ckpt_dir, '*.pth')))
for f in files:
    print(os.path.basename(f))

net_g_10000.pth
net_g_2000.pth
net_g_4000.pth
net_g_6000.pth
net_g_8000.pth
net_g_latest.pth


In [20]:
import glob, os

ckpt_dir = '/content/drive/MyDrive/MambaIR/experiments/MambaIRScratch-SIDL-scratch-dim48-V2/models'
files = sorted(glob.glob(os.path.join(ckpt_dir, '*.pth')))
for f in files:
    print(os.path.basename(f))

net_g_10000.pth
net_g_12000.pth
net_g_14000.pth
net_g_16000.pth
net_g_18000.pth
net_g_2000.pth
net_g_20000.pth
net_g_22000.pth
net_g_24000.pth
net_g_26000.pth
net_g_28000.pth
net_g_30000.pth
net_g_32000.pth
net_g_34000.pth
net_g_36000.pth
net_g_38000.pth
net_g_4000.pth
net_g_40000.pth
net_g_42000.pth
net_g_44000.pth
net_g_46000.pth
net_g_48000.pth
net_g_50000.pth
net_g_52000.pth
net_g_54000.pth
net_g_56000.pth
net_g_58000.pth
net_g_6000.pth
net_g_60000.pth
net_g_62000.pth
net_g_64000.pth
net_g_66000.pth
net_g_68000.pth
net_g_70000.pth
net_g_72000.pth
net_g_74000.pth
net_g_8000.pth


In [ ]:
0

## 클로드 ver


In [21]:
import os

task = 'scratch'
width = 48
num_blocks = 8
difficulty_levels = ['easy', 'medium', 'hard']

ckpt_path = '/content/drive/MyDrive/MambaIR/experiments/MambaIRScratch-SIDL-scratch-dim48-V2/models/net_g_74000.pth'

out_dir = '/content/drive/MyDrive/sidl_options/test'
os.makedirs(out_dir, exist_ok=True)

for difficulty in difficulty_levels:
    template = f"""\
name: MambaIRScratch-SIDL-{task}-{difficulty}-test
model_type: SRModel
scale: 1
num_gpu: 1
manual_seed: 42
path:
  root: /content/drive/MyDrive/MambaIR/results
  pretrain_network_g: {ckpt_path}
  strict_load_g: true
  resume_state: ~
datasets:
  test:
    name: sidl-{task}-{difficulty}-test
    type: PairedImageDataset
    dataroot_gt: /content/drive/MyDrive/test/{task}/{difficulty}/input
    dataroot_lq: /content/drive/MyDrive/test/{task}/{difficulty}/input
    io_backend:
      type: disk
network_g:
  type: MambaIRScratch
  inp_channels: 3
  out_channels: 3
  dim: {width}
  num_blocks: {num_blocks}
  d_state: 16
  d_conv: 4
  expand: 2
  img_range: 1.0
val:
  save_img: true
  suffix: ~
  metrics:
    psnr:
      type: calculate_psnr
      crop_border: 0
      test_y_channel: false
    ssim:
      type: calculate_ssim
      crop_border: 0
      test_y_channel: false
dist_params:
  backend: nccl
  port: 29500
"""
    out_path = os.path.join(out_dir, f'MambaIRScratch_{task}_{difficulty}.yml')
    with open(out_path, 'w') as f:
        f.write(template)
    print(f'생성: {out_path}')

생성: /content/drive/MyDrive/sidl_options/test/MambaIRScratch_scratch_easy.yml
생성: /content/drive/MyDrive/sidl_options/test/MambaIRScratch_scratch_medium.yml
생성: /content/drive/MyDrive/sidl_options/test/MambaIRScratch_scratch_hard.yml


In [23]:
import os

# 1. torchvision 패치
import site
for path in site.getsitepackages():
    target = os.path.join(path, 'basicsr/data/degradations.py')
    if os.path.exists(target):
        with open(target, 'r') as f:
            content = f.read()
        content = content.replace(
            'from torchvision.transforms.functional_tensor import rgb_to_grayscale',
            'from torchvision.transforms.functional import rgb_to_grayscale'
        )
        with open(target, 'w') as f:
            f.write(content)
        print(f'torchvision 패치 완료: {target}')
        break

# 2. pip basicsr 제거 후 MambaIR 레포 버전으로
!pip uninstall basicsr -y
os.chdir('/content/drive/MyDrive/MambaIR')
!python setup.py develop --no_cuda_ext 2>/dev/null || echo "setup 완료"

# 3. base_model 패치 (CosineAnnealingLR)
path = '/content/drive/MyDrive/MambaIR/basicsr/models/base_model.py'
with open(path, 'r') as f:
    content = f.read()
old = """        else:
            if scheduler_type == 'CosineAnnealingLR':
                schedulers.append(
                    CosineAnnealingLR(
                        self.optimizer_g,
                        T_max=train_opt['scheduler']['T_max'],
                        eta_min=train_opt['scheduler'].get('eta_min', 0)
                    )
                )
            else:
                raise NotImplementedError(f'Scheduler {scheduler_type} is not implemented yet.')"""
new = """        elif scheduler_type == 'CosineAnnealingLR':
            for optimizer in self.optimizers:
                self.schedulers.append(
                    torch.optim.lr_scheduler.CosineAnnealingLR(
                        optimizer,
                        T_max=train_opt['scheduler']['T_max'],
                        eta_min=train_opt['scheduler'].get('eta_min', 0)
                    )
                )
        else:
            raise NotImplementedError(f'Scheduler {scheduler_type} is not implemented yet.')"""
if old in content:
    content = content.replace(old, new)
    with open(path, 'w') as f:
        f.write(content)
    print('base_model 패치 완료')
else:
    print('이미 패치됨 or 스킵')

print('준비 완료!')

torchvision 패치 완료: /usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py
Found existing installation: basicsr 1.4.2
Uninstalling basicsr-1.4.2:
  Successfully uninstalled basicsr-1.4.2
setup 완료
이미 패치됨 or 스킵
준비 완료!


In [25]:
# archs __init__.py 확인
with open('/content/drive/MyDrive/MambaIR/basicsr/archs/__init__.py', 'r') as f:
    print(f.read())

import importlib
from copy import deepcopy
from os import path as osp

from basicsr.utils import get_root_logger, scandir
from basicsr.utils.registry import ARCH_REGISTRY

__all__ = ['build_network']

# automatically scan and import arch modules for registry
# scan all the files under the 'archs' folder and collect files ending with
# '_arch.py'
arch_folder = osp.dirname(osp.abspath(__file__))
arch_filenames = [osp.splitext(osp.basename(v))[0] for v in scandir(arch_folder) if v.endswith('_arch.py')]
# import all the arch modules
_arch_modules = [importlib.import_module(f'basicsr.archs.{file_name}') for file_name in arch_filenames]


def build_network(opt):
    opt = deepcopy(opt)
    network_type = opt.pop('type')
    net = ARCH_REGISTRY.get(network_type)(**opt)
    logger = get_root_logger()
    logger.info(f'Network [{net.__class__.__name__}] is created.')
    return net

from .mambair_scratch_arch import MambaIRScratch



In [26]:
path = '/content/drive/MyDrive/MambaIR/basicsr/archs/__init__.py'
with open(path, 'r') as f:
    content = f.read()

content = content.replace('\nfrom .mambair_scratch_arch import MambaIRScratch\n', '')

with open(path, 'w') as f:
    f.write(content)

print('완료')

# 확인
with open(path, 'r') as f:
    print(f.read())

완료
import importlib
from copy import deepcopy
from os import path as osp

from basicsr.utils import get_root_logger, scandir
from basicsr.utils.registry import ARCH_REGISTRY

__all__ = ['build_network']

# automatically scan and import arch modules for registry
# scan all the files under the 'archs' folder and collect files ending with
# '_arch.py'
arch_folder = osp.dirname(osp.abspath(__file__))
arch_filenames = [osp.splitext(osp.basename(v))[0] for v in scandir(arch_folder) if v.endswith('_arch.py')]
# import all the arch modules
_arch_modules = [importlib.import_module(f'basicsr.archs.{file_name}') for file_name in arch_filenames]


def build_network(opt):
    opt = deepcopy(opt)
    network_type = opt.pop('type')
    net = ARCH_REGISTRY.get(network_type)(**opt)
    logger = get_root_logger()
    logger.info(f'Network [{net.__class__.__name__}] is created.')
    return net



In [28]:
path = '/content/drive/MyDrive/MambaIR/basicsr/archs/__init__.py'

new_content = """import importlib
from copy import deepcopy
from os import path as osp

from basicsr.utils import get_root_logger, scandir
from basicsr.utils.registry import ARCH_REGISTRY

__all__ = ['build_network']

arch_folder = osp.dirname(osp.abspath(__file__))
arch_filenames = [osp.splitext(osp.basename(v))[0] for v in scandir(arch_folder) if v.endswith('_arch.py')]
_arch_modules = [importlib.import_module(f'basicsr.archs.{file_name}') for file_name in arch_filenames]


def build_network(opt):
    opt = deepcopy(opt)
    network_type = opt.pop('type')
    net = ARCH_REGISTRY.get(network_type)(**opt)
    logger = get_root_logger()
    logger.info(f'Network [{net.__class__.__name__}] is created.')
    return net
"""

with open(path, 'w') as f:
    f.write(new_content)

print('완료')
with open(path, 'r') as f:
    print(f.read())

완료
import importlib
from copy import deepcopy
from os import path as osp

from basicsr.utils import get_root_logger, scandir
from basicsr.utils.registry import ARCH_REGISTRY

__all__ = ['build_network']

arch_folder = osp.dirname(osp.abspath(__file__))
arch_filenames = [osp.splitext(osp.basename(v))[0] for v in scandir(arch_folder) if v.endswith('_arch.py')]
_arch_modules = [importlib.import_module(f'basicsr.archs.{file_name}') for file_name in arch_filenames]


def build_network(opt):
    opt = deepcopy(opt)
    network_type = opt.pop('type')
    net = ARCH_REGISTRY.get(network_type)(**opt)
    logger = get_root_logger()
    logger.info(f'Network [{net.__class__.__name__}] is created.')
    return net



In [30]:
# arch 파일 196번째 줄 근처 확인
with open('/content/drive/MyDrive/MambaIR/basicsr/archs/mambair_scratch_arch.py', 'r') as f:
    lines = f.readlines()
# 190~200번째 줄 출력
for i, line in enumerate(lines[188:205], start=189):
    print(f'{i}: {line}', end='')

189: # 6. MambaIRScratch (최종 네트워크)
190: #    MambaIR 전체 구조 + ScratchMambaBlock 적용
191: # ─────────────────────────────────────────────
192: import torch
193: import torch.nn as nn
194: from basicsr.utils.registry import ARCH_REGISTRY
195: 
196: @ARCH_REGISTRY.register()
197: 
198: # (기존 블록 import 코드 등 유지...)
199: 
200: @ARCH_REGISTRY.register()
201: class MambaIRScratch(nn.Module):
202:     def __init__(self, inp_channels=3, out_channels=3, dim=48, num_blocks=8, d_state=16, d_conv=4, expand=2, img_range=1.):
203:         super(MambaIRScratch, self).__init__()
204:         self.img_range = img_range
205:         self.num_blocks = num_blocks


In [31]:
import os
print(os.listdir('/content/drive/MyDrive/MambaIR/basicsr/archs/'))

['arch_util.py', 'mambair_arch.py', 'mambairv2_arch.py', 'mambairv2light_arch.py', '__init__.py', '__pycache__', 'mambair_scratch_arch.py']


In [32]:
path = '/content/drive/MyDrive/MambaIR/basicsr/archs/mambair_scratch_arch.py'

with open(path, 'r') as f:
    content = f.read()

# 196번째 줄의 빈 register 제거
content = content.replace(
    '@ARCH_REGISTRY.register()\n\n# (기존 블록 import 코드 등 유지...)\n\n@ARCH_REGISTRY.register()\nclass MambaIRScratch',
    '@ARCH_REGISTRY.register()\nclass MambaIRScratch'
)

with open(path, 'w') as f:
    f.write(content)

# 확인
with open(path, 'r') as f:
    lines = f.readlines()
for i, line in enumerate(lines[188:205], start=189):
    print(f'{i}: {line}', end='')

189: # 6. MambaIRScratch (최종 네트워크)
190: #    MambaIR 전체 구조 + ScratchMambaBlock 적용
191: # ─────────────────────────────────────────────
192: import torch
193: import torch.nn as nn
194: from basicsr.utils.registry import ARCH_REGISTRY
195: 
196: @ARCH_REGISTRY.register()
197: class MambaIRScratch(nn.Module):
198:     def __init__(self, inp_channels=3, out_channels=3, dim=48, num_blocks=8, d_state=16, d_conv=4, expand=2, img_range=1.):
199:         super(MambaIRScratch, self).__init__()
200:         self.img_range = img_range
201:         self.num_blocks = num_blocks
202:         
203:         self.conv_in = nn.Conv2d(inp_channels, dim, 3, 1, 1)
204:         
205:         # 블록들을 리스트로 관리하여 각각의 출력을 빼낼 수 있게 분리 (Dense Skip 핵심)


In [33]:
%cd /content/drive/MyDrive/MambaIR

for diff in ['easy', 'medium', 'hard']:
    print(f'========== Testing: {diff} ==========')
    !torchrun --standalone \
      --nnodes=1 \
      --nproc_per_node=1 \
      --master_port=4321 \
      basicsr/test.py \
      -opt /content/drive/MyDrive/sidl_options/test/MambaIRScratch_scratch_{diff}.yml \
      --launcher pytorch

/content/drive/MyDrive/MambaIR
========== Testing: easy ==========
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Path already exists. Rename it to /content/drive/MyDrive/MambaIR/results/MambaIRScratch-SIDL-scratch-easy-test_archived_20260603_122101
2026-06-03 12:21:02,015 INFO: 
  name: MambaIRScratch-SIDL-scratch-easy-test
  model_type: SRModel
  scale: 1
  num_gpu: 1
  manual_seed: 42
  path:[
    root: /content/drive/MyDrive/MambaIR/results
    pretrain_network_g: /content/drive/MyDrive/MambaIR/experiments/MambaIRScratch-SIDL-scratch-dim48-V2/models/net_g_74000.pth
    strict_load_g: True
    resume_state: None
    results_root: /content/drive/MyDrive/MambaIR/results/MambaIRScratch-SIDL-scratch-easy-test
    log: /content/drive/MyDrive/MambaIR/results/MambaI

In [34]:
# 현재 arch 파일의 전체 클래스 구조 확인
with open('/content/drive/MyDrive/MambaIR/basicsr/archs/mambair_scratch_arch.py', 'r') as f:
    content = f.read()
print(content)

"""
MambaIR-Scratch: MambaIR + Scratch-specific modules
  - DirectionalSSM  : 1x7 / 7x1 방향 커널로 선형 패턴 명시적 학습
  - ScratchMaskAttn : 손상 영역 집중 처리 (Partial Conv 아이디어)
  - ScratchMambaBlock : RSSB 교체용 블록

배치 위치: basicsr/archs/mambair_scratch_arch.py
등록 방법: basicsr/archs/__init__.py 에 import 한 줄 추가
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange
from mamba_ssm import Mamba
from basicsr.utils.registry import ARCH_REGISTRY


# ─────────────────────────────────────────────
# 0. 공통 유틸
# ─────────────────────────────────────────────
class LayerNorm(nn.Module):
    """채널-우선(C,H,W) 입력용 LayerNorm"""
    def __init__(self, dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        # x: (B, C, H, W)
        x = x.permute(0, 2, 3, 1)   # → (B,H,W,C)
        x = self.norm(x)
        return x.permute(0, 3, 1, 2) # → (B,C,H,W)


# ─────────────────────────────────────────────
# 1. DirectionalConv

In [35]:
import os, shutil, glob, tarfile

task = 'scratch'
difficulty_levels = ['easy', 'medium', 'hard']
base_path = '/content/drive/MyDrive/MambaIR/results'

source_dirs = {
    diff: os.path.join(base_path, f'MambaIRScratch-SIDL-scratch-{diff}-test/visualization/sidl-scratch-{diff}-test')
    for diff in difficulty_levels
}

temp_dir = '/content/temp_submission_final'
os.makedirs(temp_dir, exist_ok=True)
task_dir = os.path.join(temp_dir, task)
os.makedirs(task_dir, exist_ok=True)

for diff in difficulty_levels:
    input_dir = os.path.join(task_dir, diff, 'input')
    os.makedirs(input_dir, exist_ok=True)
    png_files = glob.glob(os.path.join(source_dirs[diff], '*.png'))
    print(f'{diff}: {len(png_files)}개 파일 발견')
    for f_path in png_files:
        filename = os.path.basename(f_path)
        clean_name = filename.replace(f'_MambaIRScratch-SIDL-scratch-{diff}-test', '')
        shutil.copy(f_path, os.path.join(input_dir, clean_name))

tar_path = '/content/drive/MyDrive/sidl_submission_final.tar'
with tarfile.open(tar_path, 'w') as tar:
    tar.add(task_dir, arcname=task)

print(f'완료: {tar_path}')
shutil.rmtree(temp_dir)

easy: 672개 파일 발견
medium: 480개 파일 발견
hard: 768개 파일 발견
완료: /content/drive/MyDrive/sidl_submission_final.tar


In [36]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/MambaIR')

import torch
from basicsr.archs.mambair_scratch_arch import MambaIRScratch

model = MambaIRScratch(inp_channels=3, out_channels=3, dim=48, num_blocks=8).cuda()

# 파라미터 수
params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {params:,} ({params/1e6:.2f} M)')

# MACs 측정
try:
    from thop import profile
    dummy = torch.randn(1, 3, 256, 256).cuda()
    macs, _ = profile(model, inputs=(dummy,), verbose=False)
    print(f'MACs: {macs/1e9:.2f} G ({macs/1e9/2:.2f} GFLOPs)')
except:
    print('thop 없음, 설치 후 재시도')
    !pip install thop -q

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Parameters: 325,019 (0.33 M)
thop 없음, 설치 후 재시도


In [37]:
from thop import profile
import torch
from basicsr.archs.mambair_scratch_arch import MambaIRScratch

model = MambaIRScratch(inp_channels=3, out_channels=3, dim=48, num_blocks=8).cuda()
dummy = torch.randn(1, 3, 256, 256).cuda()
macs, params = profile(model, inputs=(dummy,), verbose=False)

print(f'Parameters: {params/1e6:.2f} M')
print(f'MACs: {macs/1e9:.2f} G')
print(f'GFLOPs: {macs*2/1e9:.2f} G')

Parameters: 0.17 M
MACs: 10.72 G
GFLOPs: 21.43 G
